In [ ]:
import os
import requests
from tqdm import tqdm
from pyspark.sql import SparkSession


api_url = "https://huggingface.co/api/datasets/ashraq/fashion-product-images-small/parquet/default/train"

print("--> Consultando endpoint de Hugging Face...")
res = requests.get(api_url).json()

# La API devuelve una lista con los paths; cogemos el primero
parquet_direct_url = res[0]
target_path = "data/fashion_dataset.parquet"
os.makedirs("data", exist_ok=True)

print(f"--> Descargando Parquet desde: {parquet_direct_url}")

# 2. Descargar por streaming en bloques pequeños para no saturar la RAM
response = requests.get(parquet_direct_url, stream=True)
total_size = int(response.headers.get('content-length', 0))

with open(target_path, "wb") as f, tqdm(
    desc="Progreso de descarga",
    total=total_size,
    unit='iB',
    unit_scale=True,
    unit_divisor=1024,
) as bar:
    for chunk in response.iter_content(chunk_size=8192):
        if chunk:
            size = f.write(chunk)
            bar.update(size)

print(f"\n✅ Archivo guardado localmente en: {target_path}")

# 3. Inicializar Spark y cargar el Parquet de forma eficiente
spark = SparkSession.builder \
    .appName("VintedFashionDataset") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print("--> Cargando Parquet en PySpark...")
df_spark = spark.read.parquet(target_path)

# 4. Inspeccionar el esquema y una muestra sin cargar todo en memoria
print("\n--- Esquema del Dataset ---")
df_spark.printSchema()

print("\n--- Primeras 3 filas ---")
df_spark.show(3, truncate=50)

--> Consultando endpoint de Hugging Face...
--> Descargando Parquet desde: https://huggingface.co/api/datasets/ashraq/fashion-product-images-small/parquet/default/train/0.parquet


Progreso de descarga: 100%|██████████| 130M/130M [00:05<00:00, 23.7MiB/s] 



✅ Archivo guardado localmente en: data/fashion_dataset.parquet
--> Cargando Parquet en PySpark...

--- Esquema del Dataset ---
root
 |-- id: long (nullable = true)
 |-- gender: string (nullable = true)
 |-- masterCategory: string (nullable = true)
 |-- subCategory: string (nullable = true)
 |-- articleType: string (nullable = true)
 |-- baseColour: string (nullable = true)
 |-- season: string (nullable = true)
 |-- year: double (nullable = true)
 |-- usage: string (nullable = true)
 |-- productDisplayName: string (nullable = true)
 |-- image: struct (nullable = true)
 |    |-- bytes: binary (nullable = true)
 |    |-- path: string (nullable = true)


--- Primeras 3 filas ---
+-----+------+--------------+-----------+-----------+----------+------+------+------+----------------------------------+--------------------------------------------------+
|   id|gender|masterCategory|subCategory|articleType|baseColour|season|  year| usage|                productDisplayName|                       